# START

In [ ]:

import sys
sys.path.append(r"../PySlyde")

In [2]:
import os
import glob
import json
import xml.etree.ElementTree as ET

import cv2
import pandas
import openslide
import numpy as np
import pandas as pd
import seaborn as sns
from itertools import chain
import matplotlib.pyplot as plt

from pyslyde import Slide, WSIParser, Annotations
from pyslyde.util.utilities import TissueDetect, visualise_wsi_tiling

In [3]:
from datetime import date

# Returns the current local date
today = date.today()
print("Today date is: ", today)

Today date is:  2026-02-02


In [4]:
import re

def extract_ids(filename):
    # This regex finds "LEAP" followed by digits, then later "Slide" (with an optional hyphen, underscore, or space) and digits.
    pattern = r'(SLIDE \d+).*?.ndpi'
    match = re.search(pattern, filename, re.IGNORECASE)
    #print(filename)
    if match:
        sname = match.group(0)
    else:
        sname = None
        print("no sname match")

    pattern = r'Slide \d+'
    match = re.search(pattern, filename, re.IGNORECASE)
    if match:
        snum = match.group(0)
    else:
        snum = None
        print("no slide num")

    pattern = r'LEAP\d+'
    match = re.search(pattern, filename, re.IGNORECASE)
    if match:
        leap = match.group(0)
    else:
        leap = None
        print("no LEAPID")
    
    print(f"{leap} | {snum} | {sname}")
    return sname, snum, leap

In [ ]:
root_path = '../data' 
WSI_PATH = os.path.join(root_path,'WSI','0-original')
annotation_path = os.path.join(root_path,'annotations','treatment_response','validation')
tissue_mask_path = os.path.join(root_path,'tissue_masks')

In [ ]:
STRIDE = 224
TILE_DIMS = 224
MAG_LEVEL = 2
FILTER_THRESHOLD = 0.5
VIS_PATH = '../output/tileviz'
PATCH_PATH = os.path.join(root_path,'patches','treatment_response','validation')
mask_mag = 2.5

In [ ]:
#all_annot_paths = glob.glob('*.json', root_dir=annotation_path)
all_annot_paths = glob.glob(os.path.join(annotation_path,'*.json'))
all_annot_paths = sorted(all_annot_paths)

In [ ]:

all_mask_paths = glob.glob(os.path.join(tissue_mask_path,'*.png'))
all_mask_paths = sorted(all_mask_paths)


# LOOP all slides

# Single slide START

In [ ]:
json_path =all_annot_paths[2]
slide_name,slide_num, leap_id = extract_ids(json_path)
print(f"starting {slide_num}")
print(slide_name)
slide_path = os.path.join(WSI_PATH,slide_name)

try:
    resolved_path = os.path.realpath(slide_path)
    wsi = Slide(resolved_path, annotations_path=json_path, source='qupath')
except openslide.lowlevel.OpenSlideError as e:
    err_msg = f'Failed to open WSI file {slide_path} with resolved path {resolved_path}: {e}.'
    print(err_msg)

base_mag = int(wsi.properties.get(openslide.PROPERTY_NAME_OBJECTIVE_POWER, None))


In [ ]:
mask_file = f"{slide_name}_mask_use.png"
if mask_file in (os.path.basename(f) for f in all_mask_paths):
    print("mask found!")
            
    tissue_mask = cv2.imread(os.path.join(tissue_mask_path,mask_file), cv2.IMREAD_GRAYSCALE) 
    tissue_mask = (tissue_mask > 0).astype(np.uint8)

    scale_factor = int(base_mag/mask_mag)
    tissue_mask_40x = cv2.resize(
        tissue_mask,
        None,  # automatically determine new size from scale factors
        fx=scale_factor,
        fy=scale_factor,
        interpolation=cv2.INTER_NEAREST  # keeps mask values crisp (no smoothing)
    )
    print("sf:",scale_factor)
    print("tissue_mask_40x:",tissue_mask_40x.shape)

else:
    print("mask not found")
    detector = TissueDetect(wsi)
    thumb = detector.tissue_thumbnail
    tissue_mask_40x = detector.detect_tissue()

In [ ]:

plt.imshow(cv2.resize(tissue_mask_40x,(1000,1000)),cmap='gray')
plt.show()

In [ ]:

# Initialize parser object
normalize = None
border=wsi.get_border()
parser = WSIParser(wsi, tile_dim=TILE_DIMS, border=border, mag_level=MAG_LEVEL)
num = parser.tiler(stride=STRIDE)
print(f'Tiles: {num}')

In [ ]:
# Filter tiles
parser.filter_tissue(
    tissue_mask_40x,
    label=1,
    threshold=FILTER_THRESHOLD)
print(f'Filtered tiles: {parser.number}')


In [ ]:
viz_file_path = os.path.join(VIS_PATH,
                    slide_name+'_tiling-{today}.png')

visualise_wsi_tiling(
    wsi,
    parser,
    viz_file_path,
    viewing_res=4
    )


In [ ]:
%matplotlib inline
img = cv2.imread(viz_file_path)
#plt.figure(figsize=(8,8))
plt.imshow(img)
plt.show()

In [ ]:
# Save tiles
func = parser.extract_tiles(apply_mask=False) #True, mask=tissue_mask_10x)
parser.save(func, os.path.join(PATCH_PATH,f"{slide_num}_{leap_id}"))

In [10]:
done = []

In [11]:
need_mask = []
has_mask = []


In [ ]:

for json_path in all_annot_paths:
    slide_name,slide_num, leap_id = extract_ids(json_path)
    slide_path = os.path.join(WSI_PATH,slide_name)
    # first check for an existing mask, if none exists use pyslyde to detect tissue
    mask_file = f"{slide_name}_mask_use.png"
    
    if mask_file in (os.path.basename(f) for f in all_mask_paths):
        print("mask found!")
        has_mask.append(json_path)
    else:
        print("mask not found - save for later")
        need_mask.append(json_path)

In [ ]:

for json_path in has_mask:
    slide_name,slide_num, leap_id = extract_ids(json_path)
    slide_path = os.path.join(WSI_PATH,slide_name)
    # first check for an existing mask, if none exists use pyslyde to detect tissue
    mask_file = f"{slide_name}_mask_use.png"
    if mask_file in (os.path.basename(f) for f in all_mask_paths):
        print("mask found!")
    else:
        print("mask not found - save for later")
        need_mask.append(json_path)
        continue

    try:
        resolved_path = os.path.realpath(slide_path)
        wsi = Slide(resolved_path, annotations_path=json_path, source='qupath')
    except openslide.lowlevel.OpenSlideError as e:
        err_msg = f'Failed to open WSI file {slide_path} with resolved path {resolved_path}: {e}.'
        print(err_msg)

    base_mag = int(wsi.properties.get(openslide.PROPERTY_NAME_OBJECTIVE_POWER, None))
    print(f"base mag: {base_mag}")

    # get mask

                
    tissue_mask = cv2.imread(os.path.join(tissue_mask_path,mask_file), cv2.IMREAD_GRAYSCALE) 
    tissue_mask = (tissue_mask > 0).astype(np.uint8)

    scale_factor = int(base_mag/mask_mag)
    tissue_mask_40x = cv2.resize(
        tissue_mask,
        None,  # automatically determine new size from scale factors
        fx=scale_factor,
        fy=scale_factor,
        interpolation=cv2.INTER_NEAREST  # keeps mask values crisp (no smoothing)
    )
    #print("sf:",scale_factor)
    #print("tissue_mask_40x:",tissue_mask_40x.shape)


    
    # plt.imshow(cv2.resize(tissue_mask_40x,(1000,1000)),cmap='gray')
    # plt.show()

    # Initialize parser object
    normalize = None
    border = wsi.get_border()
    parser = WSIParser(wsi, tile_dim=TILE_DIMS, border=border, mag_level=MAG_LEVEL)
    num = parser.tiler(stride=STRIDE)
    print(f'Tiles: {num}')
   

    # Filter tiles
    parser.filter_tissue(
        tissue_mask_40x,
        label=1,
        threshold=FILTER_THRESHOLD)
    print(f'Filtered tiles: {parser.number}')


    viz_file_path = os.path.join(VIS_PATH,
                    f'{slide_name}_{leap_id}_tiling-{today}.png')

    visualise_wsi_tiling(
        wsi,
        parser,
        viz_file_path,
        viewing_res=4
        )
    
    # img = cv2.imread(viz_file_path)
    # #plt.figure(figsize=(8,8))
    # plt.imshow(img)
    # plt.show()

    # Save tiles
    func = parser.extract_tiles(apply_mask=False) #True, mask=tissue_mask_10x)
    parser.save(func, os.path.join(PATCH_PATH,f"{slide_num}_{leap_id}"))

    done.append(json_path)

            

In [ ]:

for json_path in all_annot_paths:
    slide_name,slide_num, leap_id = extract_ids(json_path)
    slide_path = os.path.join(WSI_PATH,slide_name)

    try:
        resolved_path = os.path.realpath(slide_path)
        wsi = Slide(resolved_path, annotations_path=json_path, source='qupath')
    except openslide.lowlevel.OpenSlideError as e:
        err_msg = f'Failed to open WSI file {slide_path} with resolved path {resolved_path}: {e}.'
        print(err_msg)

    base_mag = int(wsi.properties.get(openslide.PROPERTY_NAME_OBJECTIVE_POWER, None))
    print(f"base mag: {base_mag}")

    # get mask
    # first check for an existing mask, if none exists use pyslyde to detect tissue
    mask_file = f"{slide_name}_mask_use.png"
    if mask_file in (os.path.basename(f) for f in all_mask_paths):
        print("mask found!")
                
        tissue_mask = cv2.imread(os.path.join(tissue_mask_path,mask_file), cv2.IMREAD_GRAYSCALE) 
        tissue_mask = (tissue_mask > 0).astype(np.uint8)

        scale_factor = int(base_mag/mask_mag)
        tissue_mask_40x = cv2.resize(
            tissue_mask,
            None,  # automatically determine new size from scale factors
            fx=scale_factor,
            fy=scale_factor,
            interpolation=cv2.INTER_NEAREST  # keeps mask values crisp (no smoothing)
        )
        print("sf:",scale_factor)
        print("tissue_mask_40x:",tissue_mask_40x.shape)

    else:
        print("mask not found - using TissueDetect")
        detector = TissueDetect(wsi)
        thumb = detector.tissue_thumbnail
        tissue_mask_40x = detector.detect_tissue()



    # Initialize parser object
    normalize = None
    border = wsi.get_border()
    parser = WSIParser(wsi, tile_dim=TILE_DIMS, border=border, mag_level=MAG_LEVEL)
    num = parser.tiler(stride=STRIDE)
    print(f'Tiles: {num}')
   

    # Filter tiles
    parser.filter_tissue(
        tissue_mask_40x,
        label=1,
        threshold=FILTER_THRESHOLD)
    print(f'Filtered tiles: {parser.number}')


    viz_file_path = os.path.join(VIS_PATH,
                    f'{slide_name}_{leap_id}_tiling-{today}.png')

    visualise_wsi_tiling(
        wsi,
        parser,
        viz_file_path,
        viewing_res=4
        )
    
    # Save tiles
    func = parser.extract_tiles(apply_mask=False) #True, mask=tissue_mask_10x)
    parser.save(func, os.path.join(PATCH_PATH,f"{slide_num}_{leap_id}"))

                

In [25]:
subfolders = [f for f in glob.glob(os.path.join(PATCH_PATH,'', "*")) if os.path.isdir(f)]
print(subfolders)

['F:/01-data/LEAP\\patches\\treatment_response\\validation\\SLIDE 10_LEAP023', 'F:/01-data/LEAP\\patches\\treatment_response\\validation\\SLIDE 16_LEAP034', 'F:/01-data/LEAP\\patches\\treatment_response\\validation\\SLIDE 19_LEAP042', 'F:/01-data/LEAP\\patches\\treatment_response\\validation\\SLIDE 19_LEAP043', 'F:/01-data/LEAP\\patches\\treatment_response\\validation\\SLIDE 21_LEAP046', 'F:/01-data/LEAP\\patches\\treatment_response\\validation\\SLIDE 14_LEAP030', 'F:/01-data/LEAP\\patches\\treatment_response\\validation\\SLIDE 3_LEAP005', 'F:/01-data/LEAP\\patches\\treatment_response\\validation\\SLIDE 30_LEAP069', 'F:/01-data/LEAP\\patches\\treatment_response\\validation\\SLIDE 32_LEAP073', 'F:/01-data/LEAP\\patches\\treatment_response\\validation\\SLIDE 33_LEAP075', 'F:/01-data/LEAP\\patches\\treatment_response\\validation\\SLIDE 33_LEAP076', 'F:/01-data/LEAP\\patches\\treatment_response\\validation\\SLIDE 38_LEAP088', 'F:/01-data/LEAP\\patches\\treatment_response\\validation\\SLIDE

In [ ]:
for dirpath, dirnames, filenames in os.walk(PATCH_PATH):
    # Skip the root itself if you only want its subfolders' files
    if dirpath == PATCH_PATH:
        continue

    folder_name = os.path.basename(dirpath)

    for filename in filenames:
        old_path = os.path.join(dirpath, filename)

        # Skip if it's already prefixed with the folder name + underscore
        if filename.startswith(folder_name + "_"):
            continue

        new_filename = f"{folder_name}_{filename}"
        new_path = os.path.join(dirpath, new_filename)

        # Sanity check: avoid overwriting an existing file
        if os.path.exists(new_path):
            #print(f"[SKIP] {old_path} -> {new_path} (target exists)")
            continue

        #print(f"[RENAME] {old_path} -> {new_path}")
        os.rename(old_path, new_path)

In [ ]:
for dirpath, dirnames, filenames in os.walk(subfolders[0]):
    # Skip the root itself if you only want its subfolders' files
    if dirpath == PATCH_PATH:
        continue

    folder_name = os.path.basename(dirpath)

    for filename in filenames:
        old_path = os.path.join(dirpath, filename)

        # Skip if it's already prefixed with the folder name + underscore
        if filename.startswith(folder_name + "_"):
            continue

        new_filename = f"{folder_name}_{filename}"
        new_path = os.path.join(dirpath, new_filename)

        # Sanity check: avoid overwriting an existing file
        if os.path.exists(new_path):
            print(f"[SKIP] {old_path} -> {new_path} (target exists)")
            continue

        print(f"[RENAME] {old_path} -> {new_path}")
        os.rename(old_path, new_path)

# Get Masks

In [ ]:
tissue_mask = cv2.imread(os.path.join(tissue_mask_path,slide_name+'_mask_use.png'), cv2.IMREAD_GRAYSCALE) 
tissue_mask = (tissue_mask > 0).astype(np.uint8)

In [ ]:
base_mag = int(wsi.properties.get(openslide.PROPERTY_NAME_OBJECTIVE_POWER, None))
base_mag

In [ ]:
mask_mag = 2.5
int(base_mag/mask_mag)

In [ ]:
scale_factor = int(base_mag/mask_mag)
tissue_mask_40x = cv2.resize(
    tissue_mask,
    None,  # automatically determine new size from scale factors
    fx=scale_factor,
    fy=scale_factor,
    interpolation=cv2.INTER_NEAREST  # keeps mask values crisp (no smoothing)
)

In [ ]:
scale_factor = int(10/mask_mag)
tissue_mask_10x = cv2.resize(
    tissue_mask,
    None,  # automatically determine new size from scale factors
    fx=scale_factor,
    fy=scale_factor,
    interpolation=cv2.INTER_NEAREST  # keeps mask values crisp (no smoothing)
)

In [ ]:
%matplotlib inline
plt.figure(figsize=(7,7))
plt.imshow(cv2.resize(tissue_mask_40x,(1000,1000)))

# Load Border

In [ ]:
STRIDE = 224
TILE_DIMS = 224
MAG_LEVEL = 2

#normalizer = StainNormalizer(sn_target, sn_method)
normalize = None
border=wsi.get_border()#borders[-1]



In [ ]:
parser = WSIParser(wsi, tile_dim=TILE_DIMS, border=border, mag_level=MAG_LEVEL)

In [ ]:
num = parser.tiler(stride=STRIDE)
print(f'Tiles: {num}')

# Filter Tissue

In [ ]:
# this means that each tile has to have a minimum of 10% tissue
FILTER_THRESHOLD = 0.1
FILTER_THRESHOLD * (parser._x_dim * parser._y_dim)

In [ ]:
parser.filter_tissue(
    tissue_mask_40x,#tis_mask, 
    label=1,
    threshold=FILTER_THRESHOLD)
print(f'Filtered tiles: {parser.number}')

# Visualize

In [ ]:
VIS_PATH = '../output'
NAME = 'leap_test'

visualise_wsi_tiling(
        wsi,
        parser,
        os.path.join(VIS_PATH,
                     NAME+'_tiling.png'),
        viewing_res=4
        )

In [ ]:
%matplotlib inline
img = cv2.imread(os.path.join(VIS_PATH,'leap_test_tiling.png'))
plt.figure(figsize=(7,7))
plt.imshow(img)
plt.show()

# Save Patches

In [ ]:
patch_path = '..data/patches/treatment_response'
func = parser.extract_tiles(apply_mask=False, mask=tissue_mask_10x)
parser.save(func, os.path.join(PATCH_PATH,f"{slide_num}_{leap_id}")) 